In [1]:
# ================================================================
# URBAN BIKE-SHARING DEMAND PREDICTION
# RANDOM FOREST REGRESSOR TRAINING
# Using 53 leakage-audited features
# ================================================================

import os
import time
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")


In [2]:
# ================================================================
# PATHS & CONFIGURATION
# ================================================================

def find_file(filename, search_subpaths=None):
    candidates = [
        filename,
        os.path.join(os.getcwd(), filename),
        os.path.join('/content', filename),
        os.path.join(os.path.dirname(os.getcwd()), filename)
    ]
    if search_subpaths:
        for sp in search_subpaths:
            candidates.extend([
                sp,
                os.path.join(os.getcwd(), sp),
                os.path.join('/content', sp),
                os.path.join('..', sp),
                os.path.join(os.path.dirname(os.getcwd()), sp),
                os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), sp)
            ])
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return filename


DATASET_PATH = find_file("station_hourly_demand_features.csv", [
    "data/features/station_hourly_demand_features.csv",
    "features/station_hourly_demand_features.csv"
])

FEATURE_LIST_PATH = find_file("final_feature_list.csv", [
    "data/model_results/leakage_audit/final_feature_list.csv",
    "final_feature_list.csv",
    "feature_list.csv",
    "data/features/feature_list.csv"
])

MODEL_DIR = "models"
RESULT_DIR = os.path.join("data", "model_results", "random_forest")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "random_forest_final_model.pkl")
RESULT_PATH = os.path.join(RESULT_DIR, "random_forest_results.csv")
PREDICTION_PATH = os.path.join(RESULT_DIR, "random_forest_test_predictions.csv")
IMPORTANCE_PATH = os.path.join(RESULT_DIR, "random_forest_feature_importance.csv")
METADATA_PATH = os.path.join(RESULT_DIR, "random_forest_metadata.csv")

TARGET_COLUMN = "demand"
TIMESTAMP_COLUMN = "hour_timestamp"
STATION_COLUMN = "start_station_id"
EXPECTED_FEATURE_COUNT = 54


In [3]:
# ================================================================
# HEADER
# ================================================================

print("=" * 70)
print("URBAN BIKE-SHARING DEMAND PREDICTION")
print("RANDOM FOREST MODEL TRAINING")
print("=" * 70)


URBAN BIKE-SHARING DEMAND PREDICTION
RANDOM FOREST MODEL TRAINING


In [4]:
# ================================================================
# 1. LOAD DATASET
# ================================================================

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}. Please run feature_engineering.ipynb first or upload station_hourly_demand_features.csv.")

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")


Dataset loaded successfully.
Rows    : 1,230,810
Columns : 65


In [5]:
# ================================================================
# 2. CHECK REQUIRED COLUMNS
# ================================================================

print("\n2. CHECKING REQUIRED COLUMNS")
print("-" * 70)

required_columns = [
    TIMESTAMP_COLUMN,
    TARGET_COLUMN,
    STATION_COLUMN
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("All required columns are available.")



2. CHECKING REQUIRED COLUMNS
----------------------------------------------------------------------
All required columns are available.


In [6]:
# ================================================================
# CONVERT TIMESTAMP
# ================================================================

print("\nCONVERTING TIMESTAMP")
print("-" * 70)

df[TIMESTAMP_COLUMN] = pd.to_datetime(
    df[TIMESTAMP_COLUMN],
    errors="coerce"
)

invalid_timestamps = df[TIMESTAMP_COLUMN].isna().sum()
print(f"Invalid timestamps: {invalid_timestamps}")

if invalid_timestamps > 0:
    print(f"Cleaning: Dropping {invalid_timestamps} record(s) with invalid timestamps...")
    df = df.dropna(subset=[TIMESTAMP_COLUMN]).copy()

print(f"Start: {df[TIMESTAMP_COLUMN].min()}")
print(f"End  : {df[TIMESTAMP_COLUMN].max()}")



CONVERTING TIMESTAMP
----------------------------------------------------------------------
Invalid timestamps: 0
Start: 2013-08-29 09:00:00
End  : 2015-08-31 23:00:00


In [7]:
# ================================================================
# 4. SORT DATA
# ================================================================

print("\n4. SORTING DATA")
print("-" * 70)

df = df.sort_values(
    [TIMESTAMP_COLUMN, STATION_COLUMN]
).reset_index(drop=True)

print("Data sorted chronologically by timestamp and station.")



4. SORTING DATA
----------------------------------------------------------------------
Data sorted chronologically by timestamp and station.


In [8]:
# ================================================================
# 5. TARGET ANALYSIS
# ================================================================

print("\n5. TARGET ANALYSIS")
print("-" * 70)

target = df[TARGET_COLUMN]

print(f"Target minimum : {target.min():.0f}")
print(f"Target maximum : {target.max():.0f}")
print(f"Target mean    : {target.mean():.4f}")
print(f"Target std     : {target.std():.4f}")



5. TARGET ANALYSIS
----------------------------------------------------------------------
Target minimum : 0
Target maximum : 52
Target mean    : 0.5443
Target std     : 1.5453


In [9]:
# ================================================================
# 6. LOAD FINAL AUDITED FEATURE LIST
# ================================================================

forbidden_features = [
    "demand",
    "demand_change_1h",
    "demand_change_24h",
    "demand_change_week",
    "historical_demand",
    "historical_max_demand",
    "historical_min_demand",
    "station_weekday_hour_mean",
    "station_hour_historical_mean"
]

if os.path.exists(FEATURE_LIST_PATH) and "final" in os.path.basename(FEATURE_LIST_PATH).lower():
    feature_df = pd.read_csv(FEATURE_LIST_PATH)
    col = [c for c in ["Feature", "feature", "features", "column", "feature_name"] if c in feature_df.columns]
    feature_column = col[0] if col else feature_df.columns[0]
    features = feature_df[feature_column].dropna().astype(str).tolist()
    print(f"Loaded {len(features)} audited features from: {FEATURE_LIST_PATH}")
elif os.path.exists(FEATURE_LIST_PATH):
    feature_df = pd.read_csv(FEATURE_LIST_PATH)
    col = [c for c in ["Feature", "feature", "features", "column", "feature_name"] if c in feature_df.columns]
    feature_column = col[0] if col else feature_df.columns[0]
    raw_features = feature_df[feature_column].dropna().astype(str).tolist()
    features = [f for f in raw_features if f not in forbidden_features]
    print(f"Loaded {len(raw_features)} candidate features from: {FEATURE_LIST_PATH}")
    print(f"Filtered out {len(raw_features) - len(features)} leakage features -> {len(features)} audited features ready.")
else:
    features = [
        'start_station_id', 'special_event', 'hour', 'day', 'weekday', 'month', 'year', 'day_of_year',
        'week_of_year', 'is_weekend', 'is_weekday', 'morning_peak', 'evening_peak', 'peak_hour',
        'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos',
        'day_of_year_sin', 'day_of_year_cos', 'hour_weekday', 'weekend_hour', 'weekday_hour', 'month_hour',
        'lag_1h', 'lag_2h', 'lag_3h', 'lag_6h', 'lag_12h', 'lag_24h', 'lag_48h', 'lag_72h', 'lag_168h',
        'same_hour_previous_day', 'same_hour_previous_2days', 'same_hour_previous_3days', 'same_hour_previous_week',
        'rolling_mean_3h', 'rolling_mean_6h', 'rolling_mean_12h', 'rolling_mean_24h', 'rolling_mean_168h',
        'rolling_max_24h', 'rolling_max_168h', 'rolling_min_24h', 'rolling_min_168h',
        'rolling_std_24h', 'rolling_std_168h', 'rolling_median_24h',
        'lag_1h_ratio_24h', 'lag_24h_ratio_week', 'rolling_ratio_24h_168h'
    ]
    print(f"Loaded authoritative 54 audited features directly.")

# Ensure all forbidden features are strictly filtered out
features = [f for f in features if f not in forbidden_features]

print(f"Features ready for training: {len(features)}")


Loaded 54 audited features from: c:\Users\bella\OneDrive\Desktop\Urban bike sharing demand prediction\colab_notebooks\final_feature_list.csv
Features ready for training: 54


In [10]:
# ================================================================
# 7. VERIFY FINAL FEATURE COUNT & EXCLUSIONS
# ================================================================

print("\n7. VERIFYING FINAL FEATURE SET")
print("-" * 70)

if len(features) != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features, but found {len(features)}."
    )

forbidden_features = [
    "demand",
    "demand_change_1h",
    "demand_change_24h",
    "demand_change_week",
    "historical_demand",
    "historical_max_demand",
    "historical_min_demand",
    "station_weekday_hour_mean",
    "station_hour_historical_mean"
]

found_forbidden = [f for f in forbidden_features if f in features]
if found_forbidden:
    raise ValueError(f"Forbidden leakage features detected: {found_forbidden}")

print(f"Final audited feature count: {len(features)}")



7. VERIFYING FINAL FEATURE SET
----------------------------------------------------------------------
Final audited feature count: 54


In [11]:
# ================================================================
# 8. TARGET LEAKAGE CHECK
# ================================================================

print("\n8. TARGET LEAKAGE CHECK")
print("-" * 70)

if TARGET_COLUMN in features:
    raise ValueError("TARGET LEAKAGE DETECTED: demand is inside features.")

print("Target demand inside feature list: FALSE")
print("No direct target leakage detected.")



8. TARGET LEAKAGE CHECK
----------------------------------------------------------------------
Target demand inside feature list: FALSE
No direct target leakage detected.


In [12]:
# ================================================================
# 9. CURRENT / FUTURE DEMAND CHECK
# ================================================================

print("\n9. CURRENT / FUTURE DEMAND FEATURE CHECK")
print("-" * 70)

dangerous_terms = [
    "current_demand",
    "future_demand",
    "target_demand",
    "next_hour_demand",
    "future",
    "current"
]

risky_names = []
for feature in features:
    feature_lower = feature.lower()
    for term in dangerous_terms:
        if term in feature_lower:
            risky_names.append(feature)
            break

if risky_names:
    print(f"Potential current/future demand features: {risky_names}")
    raise ValueError("Potential current/future demand leakage detected.")

print("No current/future-demand feature names detected.")



9. CURRENT / FUTURE DEMAND FEATURE CHECK
----------------------------------------------------------------------
No current/future-demand feature names detected.


In [13]:
# ================================================================
# 10. FEATURE AVAILABILITY CHECK
# ================================================================

print("\n10. CHECKING FEATURE AVAILABILITY")
print("-" * 70)

missing_features = [f for f in features if f not in df.columns]
if missing_features:
    raise ValueError(f"Some final audited features are missing: {missing_features}")

print("All final audited features are available.")



10. CHECKING FEATURE AVAILABILITY
----------------------------------------------------------------------
All final audited features are available.


In [14]:
# ================================================================
# 11. MISSING VALUE CHECK & SELF-HEALING IMPUTATION
# ================================================================

print("\n11. CHECKING MISSING VALUES")
print("-" * 70)

missing_values = df[features].isna().sum().sum()
print(f"Total missing feature values: {missing_values}")

if missing_values > 0:
    print(f"Self-healing: Imputing {missing_values} missing feature value(s)...")
    num_cols = [c for c in features if df[c].dtype in ['int64', 'float64', 'int32', 'float32', 'int8', 'uint8']]
    cat_cols = [c for c in features if c not in num_cols]
    if num_cols:
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    for c in cat_cols:
        df[c] = df[c].fillna("No Event")
    print(f"Remaining missing feature values: {df[features].isna().sum().sum()}")

print("No missing feature values.")



11. CHECKING MISSING VALUES
----------------------------------------------------------------------
Total missing feature values: 0
No missing feature values.


In [15]:
# ================================================================
# 12. DUPLICATE STATION-HOUR CHECK
# ================================================================

print("\n12. CHECKING STATION-HOUR DUPLICATES")
print("-" * 70)

duplicate_count = df.duplicated(
    subset=[STATION_COLUMN, TIMESTAMP_COLUMN]
).sum()

print(f"Duplicate station-hour records: {duplicate_count}")

if duplicate_count > 0:
    print(f"Cleaning: Dropping {duplicate_count} duplicate station-hour records...")
    df = df.drop_duplicates(subset=[STATION_COLUMN, TIMESTAMP_COLUMN]).copy()

print("No duplicate station-hour records.")



12. CHECKING STATION-HOUR DUPLICATES
----------------------------------------------------------------------
Duplicate station-hour records: 0
No duplicate station-hour records.


In [16]:
# ================================================================
# 13. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ================================================================

print("\n13. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT")
print("-" * 70)

unique_times = np.sort(df[TIMESTAMP_COLUMN].unique())
number_of_times = len(unique_times)

train_time_end_index = int(number_of_times * 0.70)
validation_time_end_index = int(number_of_times * 0.85)

train_end_time = unique_times[train_time_end_index - 1]
validation_start_time = unique_times[train_time_end_index]
validation_end_time = unique_times[validation_time_end_index - 1]
test_start_time = unique_times[validation_time_end_index]

train_df = df[df[TIMESTAMP_COLUMN] <= train_end_time].copy()
validation_df = df[
    (df[TIMESTAMP_COLUMN] >= validation_start_time)
    & (df[TIMESTAMP_COLUMN] <= validation_end_time)
].copy()
test_df = df[df[TIMESTAMP_COLUMN] >= test_start_time].copy()

print(f"\nTRAIN      : {len(train_df):,} rows ({train_df[TIMESTAMP_COLUMN].min()} to {train_df[TIMESTAMP_COLUMN].max()})")
print(f"VALIDATION : {len(validation_df):,} rows ({validation_df[TIMESTAMP_COLUMN].min()} to {validation_df[TIMESTAMP_COLUMN].max()})")
print(f"TEST       : {len(test_df):,} rows ({test_df[TIMESTAMP_COLUMN].min()} to {test_df[TIMESTAMP_COLUMN].max()})")



13. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
----------------------------------------------------------------------

TRAIN      : 861,560 rows (2013-08-29 09:00:00 to 2015-01-24 04:00:00)
VALIDATION : 184,590 rows (2015-01-24 05:00:00 to 2015-05-14 01:00:00)
TEST       : 184,660 rows (2015-05-14 02:00:00 to 2015-08-31 23:00:00)


In [17]:
# ================================================================
# 14. TEMPORAL LEAKAGE & OVERLAP CHECK
# ================================================================

print("\n14. TEMPORAL LEAKAGE & OVERLAP CHECK")
print("-" * 70)

train_time_max = train_df[TIMESTAMP_COLUMN].max()
validation_time_min = validation_df[TIMESTAMP_COLUMN].min()
validation_time_max = validation_df[TIMESTAMP_COLUMN].max()
test_time_min = test_df[TIMESTAMP_COLUMN].min()

if train_time_max >= validation_time_min:
    raise ValueError("Train/validation temporal overlap detected.")
if validation_time_max >= test_time_min:
    raise ValueError("Validation/test temporal overlap detected.")

train_keys = set(zip(train_df[STATION_COLUMN], train_df[TIMESTAMP_COLUMN]))
validation_keys = set(zip(validation_df[STATION_COLUMN], validation_df[TIMESTAMP_COLUMN]))
test_keys = set(zip(test_df[STATION_COLUMN], test_df[TIMESTAMP_COLUMN]))

train_val_overlap = train_keys.intersection(validation_keys)
val_test_overlap = validation_keys.intersection(test_keys)
train_test_overlap = train_keys.intersection(test_keys)

if train_val_overlap or val_test_overlap or train_test_overlap:
    raise ValueError("Station-hour key overlap detected.")

print("Train -> Validation: PASSED")
print("Validation -> Test: PASSED")
print("Train -> Test: PASSED")
print("Station-hour overlap: NONE")



14. TEMPORAL LEAKAGE & OVERLAP CHECK
----------------------------------------------------------------------
Train -> Validation: PASSED
Validation -> Test: PASSED
Train -> Test: PASSED
Station-hour overlap: NONE


In [18]:
# ================================================================
# 15. PREPARE X AND y
# ================================================================

print("\n15. PREPARING MODEL DATA")
print("-" * 70)

X_train = train_df[features].copy()
y_train = train_df[TARGET_COLUMN].copy()
time_train = train_df[TIMESTAMP_COLUMN]

X_validation = validation_df[features].copy()
y_validation = validation_df[TARGET_COLUMN].copy()
time_validation = validation_df[TIMESTAMP_COLUMN]

X_test = test_df[features].copy()
y_test = test_df[TARGET_COLUMN].copy()
time_test = test_df[TIMESTAMP_COLUMN]

print(f"X_train shape     : {X_train.shape}")
print(f"y_train shape     : {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}")
print(f"y_validation shape: {y_validation.shape}")
print(f"X_test shape      : {X_test.shape}")
print(f"y_test shape      : {y_test.shape}")



15. PREPARING MODEL DATA
----------------------------------------------------------------------
X_train shape     : (861560, 54)
y_train shape     : (861560,)
X_validation shape: (184590, 54)
y_validation shape: (184590,)
X_test shape      : (184660, 54)
y_test shape      : (184660,)


In [19]:
# ================================================================
# 16. DEFINE RANDOM FOREST MODEL
# ================================================================

print("\n16. DEFINING RANDOM FOREST REGRESSOR")
print("-" * 70)

model_parameters = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1
}

for key, value in model_parameters.items():
    print(f"{key:20s}: {value}")



16. DEFINING RANDOM FOREST REGRESSOR
----------------------------------------------------------------------
n_estimators        : 200
max_depth           : 15
min_samples_split   : 10
min_samples_leaf    : 2
max_features        : sqrt
random_state        : 42
n_jobs              : -1


In [20]:
# ================================================================
# 17. TRAIN MODEL
# ================================================================

print("\n17. TRAINING RANDOM FOREST")
print("=" * 70)

model = RandomForestRegressor(**model_parameters)

training_start = time.time()
model.fit(X_train, y_train)
training_time = time.time() - training_start

print(f"Training completed in {training_time:.2f} seconds.")



17. TRAINING RANDOM FOREST
Training completed in 44.47 seconds.


In [21]:
# ================================================================
# 18. PREDICTIONS
# ================================================================

print("\n18. GENERATING PREDICTIONS")
print("-" * 70)

prediction_start = time.time()
validation_predictions = model.predict(X_validation)
test_predictions = model.predict(X_test)
prediction_time = time.time() - prediction_start

print(f"Predictions generated in {prediction_time:.4f} seconds.")



18. GENERATING PREDICTIONS
----------------------------------------------------------------------
Predictions generated in 1.3235 seconds.


In [22]:
# ================================================================
# 19. EVALUATION
# ================================================================

print("\n19. EVALUATION METRICS")
print("-" * 70)

validation_mae = mean_absolute_error(y_validation, validation_predictions)
validation_rmse = np.sqrt(mean_squared_error(y_validation, validation_predictions))
validation_r2 = r2_score(y_validation, validation_predictions)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print(f"Validation MAE  : {validation_mae:.6f}")
print(f"Validation RMSE : {validation_rmse:.6f}")
print(f"Validation R²   : {validation_r2:.6f}")
print(f"\nTest MAE        : {test_mae:.6f}")
print(f"Test RMSE       : {test_rmse:.6f}")
print(f"Test R²         : {test_r2:.6f}")



19. EVALUATION METRICS
----------------------------------------------------------------------
Validation MAE  : 0.422730
Validation RMSE : 0.893547
Validation R²   : 0.730307

Test MAE        : 0.439206
Test RMSE       : 0.908465
Test R²         : 0.731390


In [ ]:
# ================================================================
# THREE-SET PERFORMANCE EVALUATION
# Checking for Overfitting vs. Underfitting
# ================================================================

print("\nTHREE-SET PERFORMANCE EVALUATION")
print("=" * 70)

train_predictions = model.predict(X_train)
if "validation_predictions" not in locals():
    validation_predictions = model.predict(X_validation)
if "test_predictions" not in locals():
    test_predictions = model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_predictions)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
train_r2 = r2_score(y_train, train_predictions)

val_mae = mean_absolute_error(y_validation, validation_predictions)
val_rmse = np.sqrt(mean_squared_error(y_validation, validation_predictions))
val_r2 = r2_score(y_validation, validation_predictions)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print(f"{'Dataset':<14}{'MAE':<12}{'RMSE':<12}{'R²':<10}")
print("-" * 48)
print(f"{'Training':<14}{train_mae:<12.6f}{train_rmse:<12.6f}{train_r2:<10.6f}")
print(f"{'Validation':<14}{val_mae:<12.6f}{val_rmse:<12.6f}{val_r2:<10.6f}")
print(f"{'Testing':<14}{test_mae:<12.6f}{test_rmse:<12.6f}{test_r2:<10.6f}")
print("=" * 48)



THREE-SET PERFORMANCE EVALUATION
Dataset       MAE         RMSE        R²        
------------------------------------------------
Training      0.372124    0.743818    0.738601  
Validation    0.422730    0.893547    0.730307  
Testing       0.439206    0.908465    0.731390  


In [23]:
# ================================================================
# 20. SAVE MODEL
# ================================================================

print("\n20. SAVING RANDOM FOREST MODEL")
print("-" * 70)

os.makedirs(os.path.dirname(MODEL_PATH) if os.path.dirname(MODEL_PATH) else ".", exist_ok=True)
with open(MODEL_PATH, "wb") as file:
    pickle.dump(model, file)

for extra in ["random_forest_final_model.pkl", "../models/random_forest_final_model.pkl"]:
    try:
        os.makedirs(os.path.dirname(extra) if os.path.dirname(extra) else ".", exist_ok=True)
        with open(extra, "wb") as f:
            pickle.dump(model, f)
    except Exception:
        pass

print(f"Random Forest model saved: {MODEL_PATH}")



20. SAVING RANDOM FOREST MODEL
----------------------------------------------------------------------
Random Forest model saved: models\random_forest_final_model.pkl


In [24]:
# ================================================================
# 21. SAVE TEST PREDICTIONS
# ================================================================

print("\n21. SAVING TEST PREDICTIONS")
print("-" * 70)

prediction_df = pd.DataFrame({
    "hour_timestamp": test_df[TIMESTAMP_COLUMN].values,
    "start_station_id": test_df[STATION_COLUMN].values,
    "actual_demand": y_test.values,
    "predicted_demand": test_predictions,
    "absolute_error": np.abs(y_test.values - test_predictions),
    "error": y_test.values - test_predictions
})

os.makedirs(os.path.dirname(PREDICTION_PATH) if os.path.dirname(PREDICTION_PATH) else ".", exist_ok=True)
prediction_df.to_csv(PREDICTION_PATH, index=False)

for extra in ["random_forest_test_predictions.csv", "../data/model_results/random_forest/random_forest_test_predictions.csv"]:
    try:
        os.makedirs(os.path.dirname(extra) if os.path.dirname(extra) else ".", exist_ok=True)
        prediction_df.to_csv(extra, index=False)
    except Exception:
        pass

print(f"Test predictions saved: {PREDICTION_PATH}")



21. SAVING TEST PREDICTIONS
----------------------------------------------------------------------
Test predictions saved: data\model_results\random_forest\random_forest_test_predictions.csv


In [25]:
# ================================================================
# 22. FEATURE IMPORTANCE
# ================================================================

print("\n22. FEATURE IMPORTANCE")
print("-" * 70)

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("Top 10 features:")
print(importance_df.head(10).to_string(index=False))

os.makedirs(os.path.dirname(IMPORTANCE_PATH) if os.path.dirname(IMPORTANCE_PATH) else ".", exist_ok=True)
importance_df.to_csv(IMPORTANCE_PATH, index=False)

for extra in ["random_forest_feature_importance.csv", "../data/model_results/random_forest/random_forest_feature_importance.csv"]:
    try:
        os.makedirs(os.path.dirname(extra) if os.path.dirname(extra) else ".", exist_ok=True)
        importance_df.to_csv(extra, index=False)
    except Exception:
        pass

print(f"Feature importance saved: {IMPORTANCE_PATH}")



22. FEATURE IMPORTANCE
----------------------------------------------------------------------
Top 10 features:
                 Feature  Importance
 same_hour_previous_week    0.127246
                lag_168h    0.111715
  same_hour_previous_day    0.086205
                 lag_24h    0.076087
                  lag_1h    0.045305
               peak_hour    0.039147
        lag_1h_ratio_24h    0.035656
         rolling_mean_3h    0.031732
      lag_24h_ratio_week    0.029513
same_hour_previous_2days    0.027252
Feature importance saved: data\model_results\random_forest\random_forest_feature_importance.csv


In [26]:
# ================================================================
# 23. SAVE RESULTS & METADATA
# ================================================================

print("\n23. SAVING RESULTS & METADATA")
print("-" * 70)

results_df = pd.DataFrame({
    "Model": ["Random Forest"],
    "Features": [len(features)],
    "Training Rows": [len(X_train)],
    "Validation Rows": [len(X_validation)],
    "Test Rows": [len(X_test)],
    "Validation MAE": [validation_mae],
    "Validation RMSE": [validation_rmse],
    "Validation R2": [validation_r2],
    "Test MAE": [test_mae],
    "Test RMSE": [test_rmse],
    "Test R2": [test_r2],
    "Training Time (s)": [training_time],
    "Prediction Time (s)": [prediction_time]
})

os.makedirs(os.path.dirname(RESULT_PATH) if os.path.dirname(RESULT_PATH) else ".", exist_ok=True)
results_df.to_csv(RESULT_PATH, index=False)

for extra in ["random_forest_results.csv", "../data/model_results/random_forest/random_forest_results.csv"]:
    try:
        os.makedirs(os.path.dirname(extra) if os.path.dirname(extra) else ".", exist_ok=True)
        results_df.to_csv(extra, index=False)
    except Exception:
        pass

print(f"Results saved: {RESULT_PATH}")

metadata = {
    "model_name": "Random Forest",
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "random_state": 42,
    "features_count": len(features),
    "training_rows": len(X_train),
    "validation_rows": len(X_validation),
    "test_rows": len(X_test),
    "validation_mae": float(validation_mae),
    "validation_rmse": float(validation_rmse),
    "validation_r2": float(validation_r2),
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "test_r2": float(test_r2),
    "training_time_seconds": float(training_time),
    "prediction_time_seconds": float(prediction_time)
}

os.makedirs(os.path.dirname(METADATA_PATH) if os.path.dirname(METADATA_PATH) else ".", exist_ok=True)
pd.DataFrame([metadata]).to_csv(METADATA_PATH, index=False)

for extra in ["random_forest_metadata.csv", "../data/model_results/random_forest/random_forest_metadata.csv"]:
    try:
        os.makedirs(os.path.dirname(extra) if os.path.dirname(extra) else ".", exist_ok=True)
        pd.DataFrame([metadata]).to_csv(extra, index=False)
    except Exception:
        pass

print(f"Metadata saved: {METADATA_PATH}")

print("\n" + "=" * 70)
print("RANDOM FOREST TRAINING COMPLETED SUCCESSFULLY")
print("=" * 70)



23. SAVING RESULTS & METADATA
----------------------------------------------------------------------
Results saved: data\model_results\random_forest\random_forest_results.csv


Metadata saved: data\model_results\random_forest\random_forest_metadata.csv

RANDOM FOREST TRAINING COMPLETED SUCCESSFULLY
